In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_19563/3003301750.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data_atlas = pd.read_csv(


In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 101  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

lst_sigma_tot_born = []
lst_sqrt_s = []
lst_error = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0732

max_sqrt_s = 13000
step = 100
n_points = 10000

model_params = {
    'atlas': {
        'pl':  {'mg': 0.417, 'a1': 1.563, 'a2': 2.22}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}

lst_amp_born = []
lst_sqrt_s = []
lst_sigma_tot_born = []




In [4]:
# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


# -------------------------------
# Inner integral (over phi)
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) - 
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n_points)
    return result

# -------------------------------
# Outer integral (over k)
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    return phi_integral(k, mg, a1, a2, m2_func, q, n_points)

# -------------------------------
# Double integral computation
# -------------------------------
def compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points=10000):
    result, _ = fixed_quad(
        lambda k: k_integral(k, mg, a1, a2, m2_func, q, n_points),
        0, sqrt_s_val, 
        n=n_points
    )
    return result

def born_amp(diff_T, s, epsilon, t):
    
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot_born(amp_born_value, s):
    return amp_born_value.imag / s * 0.389379323


In [5]:

def calculate_born_cross_sections(start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points=10000):
    """Calculate cross sections for all sqrt_s values"""
    
    # Generate array of sqrt_s values
    sqrt_s_values = np.arange(start_sqrt_s, max_sqrt_s + step, step)
    
    # Process each sqrt_s value
    lst_sigma_tot_born = []
    lst_sqrt_s = []
    lst_amp_born = []
    
    for sqrt_s_val in sqrt_s_values:
        # Compute the double integral
        q = 0  
        diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points)
        
        # Calculate amplitude and cross section
        s = sqrt_s_val * sqrt_s_val
        amp_born_value = born_amp(diff_T, s, epsilon, 0)
        sigma_tot_born_value = sigma_tot_born(amp_born_value, s)
        
        # Store results
        lst_sigma_tot_born.append(sigma_tot_born_value)
        lst_sqrt_s.append(sqrt_s_val)
        lst_amp_born.append(amp_born_value)
    
    return lst_sigma_tot_born, lst_sqrt_s, lst_amp_born



In [6]:
mass_model = 'pl'
ensemble = 'atlas'
m2_func = get_m2_function(mass_model)
params = model_params[ensemble][mass_model]
mg, a1, a2 = params['mg'], params['a1'], params['a2']
epsilon = epsilon_values[ensemble]

# Calculate all cross sections
lst_sigma_tot_born, lst_sqrt_s, lst_amp_born = calculate_born_cross_sections(
    start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points
)

lst_s = [val**2 for val in lst_sqrt_s]

In [7]:
n = 5
q_max = 0.1


# -------------------------------
# Full chi(s,b) including k, phi, and new q integral
# -------------------------------


def chi_integral(sqrt_s_values, mg, a1, a2, m2_func, epsilon, b, q_max):
    chi_list = []

    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        # integrand over q
        def q_integrand(q):
            t = -(q**2)

            diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q)
            return (q * j0(b * q) * born_amp(diff_T, s, epsilon, t)) / s

        # integrate real and imaginary parts separately with fixed_quad
        real_part, _ = fixed_quad(lambda q: np.real(np.vectorize(q_integrand)(q)), 0, q_max, n=n)
        imag_part, _ = fixed_quad(lambda q: np.imag(np.vectorize(q_integrand)(q)), 0, q_max, n=n)

        chi_list.append(real_part + 1j * imag_part)

    return chi_list



b_max = 10

# -------------------------------
# Eikonal amplitude A_eik(s,t)
# -------------------------------
def eikonal_amplitude(sqrt_s_values, mg, a1, a2, m2_func, epsilon, b_max):

    amp_list = []

    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        # integrand over b
        def b_integrand(b):
            chi_val = chi_integral([sqrt_s_val], mg, a1, a2, m2_func, epsilon, b, q_max)[0]
            return b * (1 - np.exp(1j * chi_val))

        # integrate real and imaginary parts separately with fixed_quad

        real_part, _ = fixed_quad(lambda b: np.real(np.vectorize(b_integrand)(b)), 0, b_max, n=n)
        imag_part, _ = fixed_quad(lambda b: np.imag(np.vectorize(b_integrand)(b)), 0, b_max, n=n)

        A_eik = 1j * s * (real_part + 1j * imag_part)
        amp_list.append(A_eik)

    return amp_list



# Example usage
lst_amp_eik = eikonal_amplitude(lst_sqrt_s, mg, a1, a2, m2_func, epsilon, b_max)



In [8]:
def sigma_tot_eik(amp, s):

    return (4 * np.pi * amp.imag * 0.389379323) / s

lst_sigma_tot_eik = [sigma_tot_eik(amp, s) for amp, s in zip(lst_amp_eik, lst_s)]
print(lst_sigma_tot_eik)

[109.06221848485633, 117.14391714482365, 122.02041255890211, 125.53873942180194, 128.29791942880166, 130.5702511994297, 132.50309466462969, 134.18538997356242, 135.6750198329854, 137.01180098690733, 138.2243134781276, 139.33377589251157, 140.35637977291557, 141.30476513071562, 142.18899073790303, 143.01719372390974, 143.7960508133256, 144.5311088056042, 145.22702644163445, 145.88775474893174, 146.516673754565, 147.11669766161523, 147.69035684245486, 148.2398625292279, 148.76715840841737, 149.2739621810294, 149.7617993454354, 150.23203088573015, 150.68587614342408, 151.12443184157243, 151.54868801663036, 151.95954144029253, 152.35780699307594, 152.74422735392338, 153.11948129542904, 153.48419082115402, 153.83892733144668, 154.18421697362172, 154.5205453036284, 154.84836136091937, 155.16808124373065, 155.48009125796656, 155.78475069827496, 156.0823943105423, 156.373334480223, 156.65786318258648, 156.93625372488236, 157.2087623052191, 157.4756294123198, 157.7370810849854, 157.993330048339

In [9]:
for s, amp, sigma in zip(lst_s, lst_amp_eik, lst_sigma_tot_eik):
    print(f"s: {s}, Amp: {amp}, Sigma_tot_eik: {sigma}")

s: 10201, Amp: 227370.60923853258j, Sigma_tot_eik: 109.06221848485633
s: 40401, Amp: 967228.550153406j, Sigma_tot_eik: 117.14391714482365
s: 90601, Amp: 2259345.9267474804j, Sigma_tot_eik: 122.02041255890211
s: 160801, Amp: 4125568.087002092j, Sigma_tot_eik: 125.53873942180194
s: 251001, Amp: 6581309.510993782j, Sigma_tot_eik: 128.29791942880166
s: 361201, Amp: 9638521.799050644j, Sigma_tot_eik: 130.5702511994297
s: 491401, Amp: 13306973.981930941j, Sigma_tot_eik: 132.50309466462969
s: 641601, Amp: 17594928.856096484j, Sigma_tot_eik: 134.18538997356242
s: 811801, Amp: 22509545.49591291j, Sigma_tot_eik: 135.6750198329854
s: 1002001, Amp: 28057138.77862063j, Sigma_tot_eik: 137.01180098690733
s: 1212201, Amp: 34243356.7064518j, Sigma_tot_eik: 138.2243134781276
s: 1442401, Amp: 41073307.02653223j, Sigma_tot_eik: 139.33377589251157
s: 1692601, Amp: 48551650.844070256j, Sigma_tot_eik: 140.35637977291557
s: 1962801, Amp: 56682673.80803614j, Sigma_tot_eik: 141.30476513071562
s: 2253001, Amp: 6

In [10]:
def add_iterative_curve(fig, x_data, y_data, 
                        curve_name:str=None, color:str='blue', line_type:str='lines+markers'):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode=line_type, 
    name=curve_name,
    line=dict(
        color=color,
        width=2),
    marker=dict(size=4))
)
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

fig_amp = go.Figure()

add_iterative_curve(fig_amp, lst_sqrt_s, np.imag(lst_amp_born), curve_name='Im(A born)', color='blue')
add_iterative_curve(fig_amp, lst_sqrt_s, np.imag(lst_amp_eik), curve_name='Im(A eikonal)', color='red')
fig_amp.update_layout(
    title='Amplitude Im vs. sqrt(s)',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Amp',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
fig_amp.show(renderer = 'browser')


fig_sigma = go.Figure()

add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_born, curve_name='sigma tot born')
add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_eik, curve_name='sigma tot eikonal', color='red')


# Add ATLAS data
fig_sigma.add_trace(go.Scatter(
    x=x_atlas,
    y=y_atlas,
    mode='markers',
    marker=dict(
        color='black',
        size=6,
        symbol='square'
    ),
    error_y=dict(
        type='data',
        array=y_error_atlas,
        visible=True
    ),
    name='ATLAS Data'
))

# Configure layout
fig_sigma.update_layout(
    title='Sigma Tot vs. sqrt(s)',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Sigma Tot [mb]',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_sigma.update_xaxes(gridcolor='lightgray')
fig_sigma.update_yaxes(gridcolor='lightgray')

fig_sigma.show(renderer = 'browser')

Opening in existing browser session.
Opening in existing browser session.
